In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 12a: Random Forest — Hyperparameter Search
# Search on DDIM synthetic FashionMNIST train/val (no test set)
# Hyperparameter grid from [CITE FASHION MNIST ARTICLE]

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import itertools
import time

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Data
IMAGE_SIZE = 28
CHANNELS   = 1
CLASSES    = 10

# Hyperparameter grid from [CITE FASHION MNIST ARTICLE]
PARAM_GRID = {
    'criterion':   ['gini', 'entropy'],
    'max_depth':   [10, 50, 100],
    'n_estimators': [10, 50, 100],
}

# Paths
PROJECT_DIR = '/content/drive/MyDrive/thesis_project'  # adjust to your Drive path
SYNTH_PATH  = os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim',
                            'fashion_mnist', 'fashion_mnist_synthetic.npz')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', 'rf', 'hp_search')
os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
# ── Cell 2: Data Loading & Preprocessing ─────────────────────────────────────
# Train on synth train, evaluate on synth val
# Test set is never touched during HP search

synth_data = np.load(SYNTH_PATH)
X_train    = synth_data['X_train']
y_train    = synth_data['y_train']
X_val      = synth_data['X_val']
y_val      = synth_data['y_val']

# Normalize and flatten
def preprocess(X):
    return X.astype('float32').reshape(len(X), -1) / 255.0

X_train_n = preprocess(X_train)
X_val_n   = preprocess(X_val)

print(f'Synth train: {X_train.shape} → flattened: {X_train_n.shape}')
print(f'Synth val:   {X_val.shape}   → flattened: {X_val_n.shape}')

In [ ]:
# ── Cell 3: Hyperparameter Search ────────────────────────────────────────────

keys   = list(PARAM_GRID.keys())
combos = list(itertools.product(*[PARAM_GRID[k] for k in keys]))
print(f'Total runs: {len(combos)}\n')

records = []

for i, values in enumerate(combos):
    params = dict(zip(keys, values))
    run_name = f"crit{params['criterion']}_depth{params['max_depth']}_nest{params['n_estimators']}"

    clf = RandomForestClassifier(
        criterion=params['criterion'],
        max_depth=params['max_depth'],
        n_estimators=params['n_estimators'],
        random_state=SEED,
        n_jobs=-1
    )

    t0 = time.time()
    clf.fit(X_train_n, y_train)
    elapsed = time.time() - t0

    train_acc = accuracy_score(y_train, clf.predict(X_train_n))
    val_acc   = accuracy_score(y_val,   clf.predict(X_val_n))

    records.append({
        'run_name':    run_name,
        'criterion':   params['criterion'],
        'max_depth':   params['max_depth'],
        'n_estimators': params['n_estimators'],
        'train_acc':   round(train_acc, 4),
        'val_acc':     round(val_acc, 4),
        'fit_time_s':  round(elapsed, 1),
    })

    print(f'[{i+1:02d}/{len(combos)}] {run_name:45s}  '
          f'train={train_acc:.4f}  val={val_acc:.4f}  ({elapsed:.1f}s)')

results_df = pd.DataFrame(records)

In [ ]:
# ── Cell 4: Results ───────────────────────────────────────────────────────────

results_ranked = results_df.sort_values('val_acc', ascending=False).reset_index(drop=True)
print('Search results ranked by val_acc:')
print(results_ranked.to_string(index=False))

best = results_ranked.iloc[0]
print(f'\nBest run: {best["run_name"]}')
print(f'  criterion:    {best["criterion"]}')
print(f'  max_depth:    {best["max_depth"]}')
print(f'  n_estimators: {best["n_estimators"]}')
print(f'  val_acc:      {best["val_acc"]}')

# Save
results_df.to_csv(os.path.join(RESULTS_DIR, 'search_results.csv'), index=False)
results_ranked.to_csv(os.path.join(RESULTS_DIR, 'search_results_ranked.csv'), index=False)

In [ ]:
# ── Cell 5: Val Accuracy Plot ─────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 6))

x     = range(len(results_ranked))
bars  = ax.bar(x, results_ranked['val_acc'], color='steelblue', alpha=0.8)

# Highlight best
bars[0].set_color('darkorange')

ax.set_xticks(x)
ax.set_xticklabels(results_ranked['run_name'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Val Accuracy')
ax.set_title('Val Accuracy — All 18 RF Search Runs (DDIM FashionMNIST)')
ax.set_ylim(0, 1)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'val_accuracy_all_runs.png'),
            dpi=150, bbox_inches='tight')
plt.close()
print(f'Results saved to: {RESULTS_DIR}')